# Snæfellsjökull: historical aerial photographs

This notebook downloads the **original aerial photo mosaics** of Snæfellsjökull from Iceland's national aerial photo archive, at their full resolution (0.5–1 m), for the years in `AERIAL_YEARS`.

These are much sharper than any satellite image of the same years, and they go back further: Landsat's first image of the area is from 1972, and true-colour satellite images only start in 1984.

## Data source

| | |
|---|---|
| **Archive** | Aerial photo archive of the Natural Science Institute of Iceland (*Náttúrufræðistofnun*), formerly the National Land Survey (*Landmælingar Íslands*) |
| **Product** | Orthomosaics: photos from one flight, corrected for terrain and camera tilt and stitched into a map-aligned GeoTIFF. Made mostly by the University of Iceland for glacier-change research. |
| **Projection** | ISN93 / Lambert 1993 (EPSG:3057), the same as the DEM and the outlines |
| **Colour** | Black and white |
| **License** | CC BY 4.0. Credit: *Aerial imagery from Náttúrufræðistofnun (formerly Landmælingar Íslands)* |
| **Catalogue** | [Historical aerial image orthomosaics](https://geoportal.gis.is/geonetwork/srv/eng/catalog.search#/metadata/4fd787a1-5f3f-43ab-8d23-280ecbe6a3fb) · browse them in [Loftmyndasjá](https://loftmyndasja.gis.is/) |

The archive publishes a footprint map (a WFS layer) with the flight date, area and download link of every mosaic. This notebook asks it which mosaics cover Snæfellsjökull, then reads **only the part of each file over the glacier**, straight from the server. The original files are 0.15–1.7 GB because some cover a whole district.

Mosaics over Snæfellsjökull:

| Year | Flight date | Pixel | Covers the default area |
| --- | --- | --- | --- |
| 1945 | 16 Sep 1945 | 0.5 m | ~80% |
| 1959 | 27 Jul 1959 | 1 m | all |
| 1979 | 22 Jul 1979 | 0.72 m | all |
| 1985 | 13 Aug 1985 | 0.72 m | all |
| 1989 | 25 Aug 1989 | 0.5 m | all |
| 1991 | 19 Jul 1991 | 0.5 m | ~90% |

Some mosaics show seams where neighbouring photos were taken in different light; that's in the originals.

The Met Office's 1945 and 1979 outlines (`../Snæfellsjökull_outlines/`) were drawn from these same photos.

## Outputs

Written to `outputs/aerial/` next to this notebook.

| Path | Contents |
|---|---|
| `snaefellsjokull_aerial_<date>_<pixel>cm.tif` | The photo mosaic at its original resolution, cut to `AREA_ISN93`. Greyscale, 8-bit, no-data `0`, ISN93 (EPSG:3057). Tiled and compressed, with overviews so GIS software opens it quickly. |
| `snaefellsjokull_aerial_<date>_preview.png` | Smaller copy at `PREVIEW_PIXEL_M` per pixel |
| `snaefellsjokull_aerial_mosaics.csv` | Every mosaic the archive has over the area: year, date, pixel size, coverage, link |
| `snaefellsjokull_aerial_years.png` | The downloaded years side by side |

With `SAVE_FULL_ORIGINALS = True` the complete original files are also saved, in `outputs/aerial/originals/`.

### Figure style

All figures use one minimal black-and-white style: white background, black text and lines, greys for secondary elements, no gridlines and no top or right axis borders. Photographs keep their own tones, because the tones are the data.

## 1. Setup

In [ ]:
# Install dependencies (run once)
%pip install -q rasterio geopandas shapely matplotlib numpy pandas pillow requests

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import requests
from IPython.display import display
from PIL import Image
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from rasterio.windows import from_bounds
from shapely.geometry import box, shape
from shapely.ops import unary_union

In [ ]:
from cycler import cycler

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "text.color": "black",
    "axes.edgecolor": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.prop_cycle": cycler(color=["black"]),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "axes.linewidth": 0.8,
    "axes.titlesize": 11,
    "axes.titlelocation": "left",
    "font.family": "sans-serif",
    "font.size": 10,
    "lines.linewidth": 1.2,
    "legend.frameon": False,
})

GREY = "#9a9a9a"
LIGHT_GREY = "#d4d4d4"

## 2. Settings

In [ ]:
AERIAL_YEARS = [1959, 1979, 1985, 1989]

# Area to cut out, in ISN93 metres (left, bottom, right, top).
# Default: the same area as the DEM, i.e. the ~1890 glacier outline plus 1 km.
AREA_ISN93 = (269760, 483450, 277670, 491750)

# A mosaic is used if it covers at least this share of the area
MIN_COVERAGE = 0.05

PREVIEW_PIXEL_M = 2
SAVE_FULL_ORIGINALS = False   # also download the complete original files (0.15–1.7 GB each)

FOOTPRINT_WFS = "https://gis.lmi.is/geoserver/loftmyndir/wfs"
FOOTPRINT_LAYER = "loftmyndir:loftmyndir_hnitsettar"
CRS = "EPSG:3057"
ATTRIBUTION = "Aerial imagery: Náttúrufræðistofnun (formerly Landmælingar Íslands), CC BY 4.0"

OUT_DIR = Path("outputs/aerial")
OUT_DIR.mkdir(parents=True, exist_ok=True)
SLUG = "snaefellsjokull"

# Read the remote GeoTIFFs efficiently over HTTP
GDAL_ENV = dict(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif",
                GDAL_HTTP_MAX_RETRY="4", GDAL_HTTP_RETRY_DELAY="2")

## 3. Which mosaics cover the area?

Some years are split over two files (e.g. 1979 and 1991 over a larger area), and some files appear twice in the footprint map. Each file is listed once, with how much of the area it covers. Files whose link doesn't work are dropped.

In [ ]:
r = requests.get(FOOTPRINT_WFS, params={
    "service": "WFS", "version": "2.0.0", "request": "GetFeature", "typeNames": FOOTPRINT_LAYER,
    "outputFormat": "application/json", "srsName": CRS, "bbox": ",".join(map(str, AREA_ISN93)) + f",{CRS}",
}, timeout=120)
r.raise_for_status()

area = box(*AREA_ISN93)
footprints = {}
for feature in r.json()["features"]:
    p = feature["properties"]
    entry = footprints.setdefault(p["url"], {"year": p["year"], "date": p["acq_date"][:10], "url": p["url"], "geoms": []})
    entry["geoms"].append(shape(feature["geometry"]))


def link_works(url):
    try:
        head = requests.head(url, timeout=60, allow_redirects=True)
        return head.ok and int(head.headers.get("Content-Length", 0)) > 10_000
    except requests.RequestException:
        return False


rows = []
for entry in footprints.values():
    footprint = unary_union(entry["geoms"])
    coverage = footprint.intersection(area).area / area.area
    if coverage < MIN_COVERAGE or not link_works(entry["url"]):
        continue
    with rasterio.Env(**GDAL_ENV), rasterio.open("/vsicurl/" + entry["url"]) as src:
        pixel = src.res[0]
    rows.append({"year": entry["year"], "date": entry["date"], "pixel_m": round(pixel, 3),
                 "coverage": round(coverage, 3), "file": entry["url"].rsplit("/", 1)[-1], "url": entry["url"]})

mosaics = pd.DataFrame(rows).sort_values(["year", "pixel_m", "coverage"], ascending=[True, True, False])
mosaics.to_csv(OUT_DIR / f"{SLUG}_aerial_mosaics.csv", index=False)
mosaics.drop(columns="url")

## 4. Download the chosen years

For each year, the output grid is the sharpest file's own pixel grid, snapped to `AREA_ISN93`, so the main file's pixels are copied exactly, with no resampling. Where a year has a second file, it only fills the gaps the first one leaves.

Each year takes a minute or two: the server sends only the rows over the glacier, but they are long rows.

In [ ]:
def year_mosaic(year):
    """Cut the area out of every file for that year, on the sharpest file's grid. Returns (array, transform)."""
    files = mosaics[mosaics["year"] == year].sort_values(["pixel_m", "coverage"], ascending=[True, False])
    with rasterio.Env(**GDAL_ENV):
        with rasterio.open("/vsicurl/" + files.iloc[0]["url"]) as main:
            res = main.res[0]
            x0, y0 = main.transform.c, main.transform.f
        # Snap the area to the main file's pixel grid
        left = x0 + np.floor((AREA_ISN93[0] - x0) / res) * res
        top = y0 - np.floor((y0 - AREA_ISN93[3]) / res) * res
        width = int(np.ceil((AREA_ISN93[2] - left) / res))
        height = int(np.ceil((top - AREA_ISN93[1]) / res))
        transform = from_origin(left, top, res, res)
        bounds = (left, top - height * res, left + width * res, top)

        grey = np.zeros((height, width), "uint8")
        filled = np.zeros((height, width), bool)
        for url in files["url"]:
            with rasterio.open("/vsicurl/" + url) as src:
                window = from_bounds(*bounds, transform=src.transform)
                data = src.read(window=window, out_shape=(src.count, height, width), boundless=True,
                                fill_value=0, resampling=Resampling.nearest)
                valid = data[-1] > 0 if src.count > 1 else data[0] > 0  # last band is alpha
                take = valid & ~filled
                grey[take] = data[0][take]
                filled |= take
    return grey, filled, transform, files


def save_original(url):
    folder = OUT_DIR / "originals"
    folder.mkdir(exist_ok=True)
    path = folder / url.rsplit("/", 1)[-1]
    if path.exists():
        return
    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 22):
                f.write(chunk)


downloaded = []
for year in AERIAL_YEARS:
    if year not in set(mosaics["year"]):
        print(f"{year}: no mosaic in the archive covers the area. Available: {sorted(set(mosaics['year']))}")
        continue
    grey, filled, transform, files = year_mosaic(year)
    date = files.iloc[0]["date"]
    res_cm = round(transform.a * 100)
    stem = f"{SLUG}_aerial_{date}_{res_cm}cm"

    with rasterio.open(OUT_DIR / f"{stem}.tif", "w", driver="GTiff", width=grey.shape[1], height=grey.shape[0],
                       count=1, dtype="uint8", crs=CRS, transform=transform, nodata=0,
                       tiled=True, blockxsize=512, blockysize=512, compress="deflate", predictor=2) as dst:
        dst.write(np.where(filled, np.maximum(grey, 1), 0), 1)
        dst.build_overviews([2, 4, 8, 16, 32], Resampling.average)
        dst.update_tags(date=date, source=ATTRIBUTION, files=", ".join(files["file"]))

    # Preview: average down to PREVIEW_PIXEL_M
    factor = max(1, round(PREVIEW_PIXEL_M / transform.a))
    preview = Image.fromarray(np.where(filled, grey, 255).astype("uint8")).resize(
        (grey.shape[1] // factor, grey.shape[0] // factor), Image.Resampling.BOX)
    preview.save(OUT_DIR / f"{SLUG}_aerial_{date}_preview.png")

    if SAVE_FULL_ORIGINALS:
        for url in files["url"]:
            save_original(url)

    downloaded.append({"year": year, "date": date, "pixel_m": transform.a, "tif": f"{stem}.tif",
                       "preview": f"{SLUG}_aerial_{date}_preview.png", "coverage": filled.mean()})
    print(f"{year}: wrote {stem}.tif ({grey.shape[1]:,} x {grey.shape[0]:,} px, {transform.a:.2f} m, "
          f"{filled.mean():.0%} of the area has photo)")

## 5. The years side by side

In [ ]:
n = len(downloaded)
if n:
    fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 4.8), squeeze=False)
    for ax, row in zip(axes[0], downloaded):
        ax.imshow(Image.open(OUT_DIR / row["preview"]), cmap="gray", vmin=0, vmax=255)
        ax.axis("off")
        ax.set_title(pd.Timestamp(row["date"]).strftime("%-d %b %Y"), fontsize=12, fontweight="bold")
        ax.text(0, -0.02, f"{row['pixel_m']:.2f} m pixels", transform=ax.transAxes, va="top", fontsize=8, color=GREY)
    fig.text(0.01, 0.01, ATTRIBUTION, fontsize=7, color=GREY)
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    fig.savefig(OUT_DIR / f"{SLUG}_aerial_years.png", dpi=150)
    plt.show()